# Visualize Learning Curves

In [ ]:
# Import required libraries
import os
import datetime
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
LINE_WIDTH = 2.5
FIG_SIZE = (8.0, 4.8)  # slightly larger graph box while keeping readability
BASE_FONT_SIZE = 20
LEGEND_FONT_SIZE = 18  # small-caps labels look larger, so keep legend slightly smaller
AXIS_LABEL_FONT_SIZE = 22  # make axis labels slightly larger
SOLID_ALPHA = 0.9
SHADED_ALPHA = 0.2
COLOR = plt.get_cmap('Dark2')  # Colormap, e.g., tab10 or Dark2

plt.rcParams.update(
    {
        'font.size': BASE_FONT_SIZE,
        'legend.fontsize': LEGEND_FONT_SIZE,
        'axes.labelsize': AXIS_LABEL_FONT_SIZE,
        'text.usetex': True,
        'axes.linewidth': LINE_WIDTH,
        'xtick.major.width': LINE_WIDTH,
        'ytick.major.width': LINE_WIDTH,
        'xtick.major.size': 2*LINE_WIDTH,
        'ytick.major.size': 2*LINE_WIDTH,
        "lines.linewidth": LINE_WIDTH,
    }
)

In [ ]:
# Save options
SAVE_FIGS = True
# SAVE_DIR = os.path.join(os.getcwd(), "figures", "bilevel_lqr")
SAVE_DIR = os.getcwd()

# Data Loading options
DATA_DIRS = [
    "../data/local/experiment/BilevelLQR_BCHG_Opt/aggregated",
    "../data/local/experiment/BilevelLQR_Baseline_Opt_onpolicy/aggregated",
    "../data/local/experiment/BilevelLQR_Baseline_Opt_offpolicy/aggregated",
    ]
INCLUDE = [
    "L_algo_BCHG_Opt_init_mean_K_1_0e+00_init_mean_K_2_0e+00_actor_update_steps_n_10_critic_update_steps_n_5_replay_buffer_size_400_",
    "L_algo_Baseline_Opt_init_mean_K_1_0e+00_init_mean_K_2_0e+00_actor_update_steps_n_10_critic_update_steps_n_5_replay_buffer_size_400_",
    "L_algo_Baseline_Opt_init_mean_K_1_0e+00_init_mean_K_2_0e+00_actor_update_steps_n_2_critic_update_steps_n_5_replay_buffer_size_1e+06_",
    ]  # Plot only data whose names contain these strings
EXCLUDE = [
    ]  # Do not plot data whose names contain these strings

# Plot options
METRICS = ['Evaluation/AverageTargetReturn']  # Metrics to visualize
X_AXIS = 'StepItr'   # Quantity used for the x-axis
MAX_X_VALUE = 500
Y_LABEL = rf"Upper-level objective"  # y-axis label
X_LABEL = rf"Outer iterations: $N$"  # x-axis label
TITLE = None
LEGEND_POSITION = {
    'loc': 'upper right',  # outer: place legend outside at a fixed position; upper/lower right/left: place legend inside the corresponding corner
    'bbox_to_anchor': None,
    'ncol': 1
}
LABELS = {
    ('BCHGD_Opt',): r"\textsc{BC-HG}",
    ('Baseline_Opt','replay_buffer_size_400'): r"\textsc{Naive-PGD (on-policy)}",
    ('Baseline_Opt','replay_buffer_size_1e+06'): r"\textsc{Naive-PGD (off-policy)}",
}
# Note: the order in the legend follows the order in LABELS
ZORDER = dict(zip(LABELS.values(), [len(LABELS)-i for i in range(len(LABELS))]))
LINESTYLES = {
    ('BCHG_Opt',): '-',
    ('Baseline_Opt', 'replay_buffer_size_400'): '--',
    ('Baseline_Opt', 'replay_buffer_size_1e+06'): (0, (2, 4, 2, 4)),
}
COLORS = {
    ('BCHG_Opt',): COLOR(2),
    ('Baseline_Opt', 'replay_buffer_size_400'): COLOR(1),
    ('Baseline_Opt', 'replay_buffer_size_1e+06'): COLOR(0),
}

In [ ]:
# Redefine plot.py functions for notebook use
def find_dirs_end_with(dir, keyword):
    found_dirs = []
    for root, dirs, files in os.walk(dir):
        for d in dirs:
            if d.endswith(keyword):
                found_dirs.append(os.path.join(root, d))
    found_dirs.sort()
    return found_dirs

def find_files_end_with(dir, keyword):
    found_files = []
    for root, dirs, files in os.walk(dir):
        for f in files:
            if f.endswith(keyword):
                found_files.append(os.path.join(root, f))
    found_files.sort()
    return found_files

## Load Data

In [ ]:
# --- Load data --- #
current_dir = os.getcwd()
DATA_DIRS = [os.path.join(current_dir, d) if not os.path.isabs(d) else d for d in DATA_DIRS]

ave_data_for_plot = {DATA_DIR: [] for DATA_DIR in DATA_DIRS}
std_data_for_plot = {DATA_DIR: [] for DATA_DIR in DATA_DIRS}
x_data_for_plot = {DATA_DIR: [] for DATA_DIR in DATA_DIRS}
label_list = {DATA_DIR: [] for DATA_DIR in DATA_DIRS}
color_id_list = {DATA_DIR: [] for DATA_DIR in DATA_DIRS}
i = 0
for DATA_DIR in DATA_DIRS:
    
    ave_sep_data_list = []
    std_sep_data_list = []

    # Search the 'average' directory
    print('Source log directory:', DATA_DIR)

    ave_dirs = find_dirs_end_with(DATA_DIR, 'average')

    if not len(ave_dirs) > 0:
        raise FileNotFoundError(f"No directory with 'average' found in {DATA_DIR}")

    for ave_dir in ave_dirs:
        ave_paths = find_files_end_with(ave_dir, 'progress_average.csv')
        # Search the 'std' directory
        std_paths = find_files_end_with(ave_dir.replace('average', 'std'), 'progress_std.csv')

        if not (len(ave_paths) > 0 and len(std_paths) > 0):
            raise FileNotFoundError(f"No progress_average.csv or progress_std.csv found in {ave_dir}")
        
        # print('Data found in:', ave_dir)

        # Load
        ave_data = [pd.read_csv(p) for p in ave_paths]  # [eval, follower, leader]
        std_data = [pd.read_csv(p) for p in std_paths]  # [eval, follower, leader]
        ave_sep_data_list.append(ave_data)
        std_sep_data_list.append(std_data)

    # --- Extract data to plot --- #

    for ave_data, std_data, ave_dir in zip(ave_sep_data_list, std_sep_data_list, ave_dirs):

        label = ave_dir.rstrip('/').split('/')[-1]  # Extract directory name

        ave_for_metrics = {}
        std_for_metrics = {}
        x_for_metrics = {}
        extracted = []
        for metric in METRICS:
            for ave, std in zip(ave_data, std_data):
                if metric in ave.columns:
                    if metric in extracted:
                        raise ValueError(f"Multiple data found for {metric} in {ave_dir}")
                    ave_for_metrics[metric] = ave[metric]
                    std_for_metrics[metric] = std[metric]
                    x_for_metrics[metric] = ave[X_AXIS]
                    extracted.append(metric)

        ave_data_for_plot[DATA_DIR].append(ave_for_metrics)
        std_data_for_plot[DATA_DIR].append(std_for_metrics)
        x_data_for_plot[DATA_DIR].append(x_for_metrics)
        label_list[DATA_DIR].append(label)
        color_id_list[DATA_DIR].append(i % COLOR.N)
        i += 1

In [ ]:
def plot_average_with_std(
        ave_data: list, std_data: list, x_data: list, metric: str,
        labels: list = None, colors: list = None, linestyles: list = None, zorders: list = None,
        solid_alpha: float = SOLID_ALPHA, shaded_alpha: float = SHADED_ALPHA,
        xlim: tuple = (0, MAX_X_VALUE), ylim: tuple = None,
        x_label: str = 'Steps', y_label: str = None,
        title: str = None, show_legend: bool = True, save_figure: bool = False, tag: str = '', file_ext: str = 'pdf'
    ):
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    labels = [i for i in range(len(ave_data))] if labels is None else labels
    colors = [COLOR(i % COLOR.N) for i in range(len(ave_data))] if colors is None else colors
    linestyles = ['-'] * len(ave_data) if linestyles is None else linestyles
    zorders = [i for i in range(len(ave_data), 0, -1)] if zorders is None else zorders
    zipped = sorted(
        zip(ave_data, std_data, x_data, labels, colors, linestyles, zorders), 
        key=lambda x: x[6], reverse=True
        )  # Sort in descending order of zorder
    for ave, std, x, label, color, linestyle, zorder in zipped:
        ax.plot(x[metric], ave[metric], label=label, color=color, 
                alpha=solid_alpha, linestyle=linestyle, zorder=zorder)
        ax.fill_between(x[metric], 
                        ave[metric] - std[metric], 
                        ave[metric] + std[metric], 
                        color=color, alpha=shaded_alpha, zorder=zorder)
    ax.set_xlabel(x_label)
    y_label = metric if y_label is None else y_label
    ax.set_ylabel(y_label)
    if title is not None:
        ax.set_title(title)
    if show_legend:
        if LEGEND_POSITION['loc'] == 'outer':
            ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        else:
            ax.legend(**LEGEND_POSITION)
    ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)
    plt.show()

    # Save
    if save_figure:
        _log_dir = os.getcwd()  # current directory
        _save_dir = SAVE_DIR if SAVE_DIR is not None else os.path.join(_log_dir, 'figures')
        os.makedirs(_save_dir, exist_ok=True)
        metric_label = metric.replace('/', '_')
        suffix = f'_{tag}' if tag != '' else ''
        ext = file_ext.lower().lstrip('.')
        if ext not in ('png', 'pdf'):
            raise ValueError(f"Unsupported file_ext: {file_ext}. Use 'png' or 'pdf'.")
        file_name = f'{metric_label}_AveStd{suffix}.{ext}'
        fig.savefig(os.path.join(_save_dir, file_name), bbox_inches='tight')
        print('Saved at', os.path.join(_save_dir, file_name))

    info = {'xlim': ax.get_xlim(), 'ylim': ax.get_ylim()}
    plt.close(fig)
    return info

def color_linestyle_label_matching(label_all, color_id_all):
    # color and linestyle matching
    colors = []
    linestyles = []
    labels = []
    for label, ci in zip(label_all, color_id_all):
        color = COLOR(ci)
        if COLORS is not None:
            for ks, c in COLORS.items():
                if all([k in label for k in ks]):
                    color = c
                    break
        colors.append(color)

        linestyle = '-'
        if LINESTYLES is not None:
            for ks, s in LINESTYLES.items():
                if all([k in label for k in ks]):
                    linestyle = s
                    break
        linestyles.append(linestyle)

        if LABELS is not None:
            for ks, l in LABELS.items():
                if all([k in label for k in ks]):
                    label = l
                    break
        elif label.endswith('_average'):
            label = label[:-len('_average')]
        labels.append(label)
    return colors, linestyles, labels

## Plot data included in INCLUDE within a single figure

In [ ]:
# --- Plot & notebook display --- #
y_lim = (-0.1, 0.5)  # y-axis display range (auto if None)
ave_data_all = []
std_data_all = []
x_data_all = []
label_all = []
color_id_all = []

for DATA_DIR in DATA_DIRS:
    for ave, std, x, label, ci in zip(
        ave_data_for_plot[DATA_DIR], 
        std_data_for_plot[DATA_DIR], 
        x_data_for_plot[DATA_DIR], 
        label_list[DATA_DIR],
        color_id_list[DATA_DIR],
        ):
        if any(inc in label for inc in INCLUDE) is False:
            continue
        if any(exc in label for exc in EXCLUDE) is True:
            continue
        ave_data_all.append(ave)
        std_data_all.append(std)
        x_data_all.append(x)
        label_all.append(label)
        color_id_all.append(ci)

if not len(label_all) > 0:
    raise ValueError("No data to plot. Please check INCLUDE and EXCLUDE filters.")

# color and linestyle matching
colors, linestyles, labels = color_linestyle_label_matching(label_all, color_id_all) 
zorders = [ZORDER.get(label, 0) for label in labels]

for metric in METRICS:
    # plot
    print(f'\n* --- Mertric: {metric} --- *')
    info = plot_average_with_std(
        ave_data=ave_data_all, 
        std_data=std_data_all,
        x_data=x_data_all,
        labels=labels,
        metric=metric,
        colors=colors,
        linestyles=linestyles,
        zorders=zorders,
        ylim=y_lim,
        x_label=X_LABEL,
        y_label=Y_LABEL,
        title=TITLE,
        show_legend=True,
        save_figure=SAVE_FIGS
        )

## Plot data of each DATA_DIR

In [ ]:
plt.rcParams.update(
    {
        "lines.linewidth": 1.5,
    }
)

ave_data_all = []
x_data_all = []
label_all = []
color_id_all = []

for DATA_DIR in DATA_DIRS:
    for ave, x, label, ci in zip(ave_data_for_plot[DATA_DIR], 
                                x_data_for_plot[DATA_DIR], 
                                label_list[DATA_DIR],
                                color_id_list[DATA_DIR]):
        ave_data_all.append(ave)
        x_data_all.append(x)
        label_all.append(label)
        color_id_all.append(ci)
colors_all, linestyles_all, _ = color_linestyle_label_matching(label_all, color_id_all)
zero_std_data_all = [{metric: [0.0] * len(d[metric])} for d in ave_data_all]

for metric in METRICS:
    print(f'\n* --- Mertric: {metric} --- *')
    # Plot all data
    info = plot_average_with_std(
        ave_data=ave_data_all, 
        std_data=zero_std_data_all,
        x_data=x_data_all,
        labels=None,
        metric=metric,
        colors=colors_all,
        linestyles=linestyles_all,
        x_label=X_LABEL,
        y_label=Y_LABEL,
        title=TITLE,
        show_legend=False,
        save_figure=SAVE_FIGS,
        tag='all'
        )

    # Plot each DATA_DIR separately
    for i, DATA_DIR in enumerate(DATA_DIRS):
        # color and linestyle matching
        colors, linestyles, labels = color_linestyle_label_matching(label_list[DATA_DIR], 
                                                                    color_id_list[DATA_DIR]) 
        zero_std_data_for_plot = [{metric: [0.0] * len(d[metric])} for d in ave_data_for_plot[DATA_DIR]]
        # plot
        print(f'\nDATA_DIR: {DATA_DIR}')

        plot_average_with_std(
            ave_data=ave_data_for_plot[DATA_DIR], 
            std_data=zero_std_data_for_plot,
            x_data=x_data_for_plot[DATA_DIR],
            labels=labels,
            metric=metric,
            colors=colors,
            linestyles=linestyles,
            xlim=info['xlim'],
            ylim=info['ylim'],
            x_label=X_LABEL,
            y_label=Y_LABEL,
            title=TITLE,
            show_legend=False,
            save_figure=SAVE_FIGS,
            tag=os.path.basename(os.path.dirname(DATA_DIR))
            )